In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import os


# ----------------------------
# 0. Rutas dinámicas
# ----------------------------
BASE_DIR = Path.cwd().parent  # ajusta si ejecutas desde otra carpeta
DATASET_DIR = BASE_DIR / "dataset"
DATASET_DIR.mkdir(exist_ok=True)

INPUT_PATH = DATASET_DIR / "datos_raw.csv"
OUTPUT_PATH = DATASET_DIR / "datos_finales.csv"


df = pd.read_csv(INPUT_PATH)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


# ----------------------------
# 1. Normalizar nombres de columnas
# ----------------------------
def limpiar_nombre_columna(c):
    c = str(c)
    c = c.replace("\u200b", "")
    c = c.replace("\xa0", " ")
    c = re.sub(r"\s+", " ", c)
    return c.strip()

df.columns = [limpiar_nombre_columna(c) for c in df.columns]


# ----------------------------
# 2. Normalizar provincia
# ----------------------------
def normalizar_provincia(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    equivalencias = {
        "Coruña, La": "La Coruña",
        "Palmas, Las": "Las Palmas",
        "Rioja, La": "La Rioja",
        "Baleares": "Islas Baleares",
        "Gerona": "Girona",
        "Lérida": "Lleida",
        "Orense": "Ourense",
        "Álava": "Araba/Álava",
        "Guipúzcoa": "Gipuzkoa",
        "Vizcaya": "Bizkaia"
    }

    return equivalencias.get(x, x)

df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)

filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
df = df[~df["Provincia"].isin(filas_basura)].copy()


# ----------------------------
# 3. Eliminar columnas basura
# ----------------------------
cols_drop = [
    c for c in df.columns
    if c.startswith("Unnamed")
    or c.startswith("Escudo")
    or c.startswith("Mapa")
    or c.startswith("Posición")
    or c.startswith("Puesto")
    or c.startswith("Provincia.")
]

df = df.drop(columns=cols_drop, errors="ignore")


# ----------------------------
# 4. Unificar columnas equivalentes
# ----------------------------
def unificar_columnas(df, nombre_base):
    cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
    if not cols:
        return df

    df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
    cols_a_borrar = [c for c in cols if c != nombre_base]
    df = df.drop(columns=cols_a_borrar, errors="ignore")
    return df


bases_a_unificar = [
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Población",
    "Presupuesto (€)",
    "Densidad (hab./km²)",
    "Superficie (km²)",
    "Código postal",
    "Código Ministerio del Interior",
    "Órgano de gobierno y administración",
    "Sede (ciudad)",
    "Extranjeros totales",
    "% de extranjeros",
    "2016", "2017", "2018", "2019",
    "2020", "2021", "2022", "2023",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie",
    "Longitud de costa (km)[2]​[3]​",
    "Superficie (km²)[2]​"
]

for base in bases_a_unificar:
    df = unificar_columnas(df, base)


# ----------------------------
# 5. Consolidar duplicados
# ----------------------------
def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# ----------------------------
# 6. Limpieza de texto
# ----------------------------
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = re.sub(r"\[\d+\]", "", x)
    x = x.replace("\xa0", " ")
    x = x.replace("\u200b", "")

    if x in ["", "-", "—", "nan", "None"]:
        return np.nan

    return x

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)


# ----------------------------
# 7. Funciones numéricas
# ----------------------------
def numero_entero_es(x):
    if pd.isna(x):
        return np.nan

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "").replace("%", "")

    if s.count(".") > 1:
        s = s.replace(".", "")
        return pd.to_numeric(s, errors="coerce")

    if s.count(".") == 1 and "," not in s:
        izq, der = s.split(".")
        if der.isdigit() and len(der) == 3:
            s = izq + der

    return pd.to_numeric(s, errors="coerce")


def numero_decimal_es(x):
    if pd.isna(x):
        return np.nan

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "").replace("%", "")

    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")

    return pd.to_numeric(s, errors="coerce")


def porcentaje_extranjeros_corregido(x):
    val = numero_decimal_es(x)
    if pd.isna(val):
        return np.nan
    return val / 10


# ----------------------------
# 8. Conversión de columnas
# ----------------------------
cols_enteras = [
    "Código postal", "Superficie (km²)", "Longitud de costa (km)[2]​[3]​",
    "Población", "Presupuesto (€)", "Extranjeros totales",
    "2016", "2017", "2018", "2019",
    "2020", "2021", "2022", "2023"
]

for col in cols_enteras:
    if col in df.columns:
        df[col] = df[col].apply(numero_entero_es)

cols_decimales = [
    "Densidad (hab./km²)", "TCAC 2016-23",
    "Porcentaje población", "Porcentaje superficie"
]

for col in cols_decimales:
    if col in df.columns:
        df[col] = df[col].apply(numero_decimal_es)

if "% de extranjeros" in df.columns:
    df["% de extranjeros"] = df["% de extranjeros"].apply(porcentaje_extranjeros_corregido)


# ----------------------------
# 9. Guardar
# ----------------------------
df.to_csv(OUTPUT_PATH, index=False)

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)
print("Shape:", df.shape)

Input: c:\Users\carlo\Tipologia y ciclo de vida de los datos\Practica1-Tipologia-y-ciclo-de-vida-de-los-datos\dataset\datos_raw.csv
Output: c:\Users\carlo\Tipologia y ciclo de vida de los datos\Practica1-Tipologia-y-ciclo-de-vida-de-los-datos\dataset\datos_finales.csv
Shape: (52, 27)


In [6]:
df_limpio = pd.read_csv(OUTPUT_PATH)
df_limpio

,Provincia,Longitud de costa (km)[2][3]_9,Superficie (km²)[2]_10,Comunidad autónoma,Capital,Ciudad más poblada*,Población,Presupuesto (€),Densidad (hab./km²),Superficie (km²),Código postal,Código Ministerio del Interior,Órgano de gobierno y administración,Sede (ciudad),Extranjeros totales,% de extranjeros,2016,2017,2018,2019,2020,2021,2022,2023,TCAC 2016-23,Porcentaje población,Porcentaje superficie
0,Albacete,NaN,14 924,Castilla-La Mancha,Albacete,Albacete,390751,1.790145e+08,26.18,NaN,2,AB,Diputación de Albacete,Palacio Provincial de Albacete (Albacete),32439,7.4,7570409,7992903,8285269,8627212,8010434,8853382,9526877,10285196,35.86,0.80,2.950
1,Alicante,244,5817,Comunidad Valenciana,Alicante,Alicante,2033566,3.138712e+08,349.59,5816.0,3,A,Diputación de Alicante,Palacio Provincial de Alicante (Alicante),456257,21.8,33899084,35422930,36463569,37807882,34503374,38470028,41939432,46472807,37.09,4.14,1.150
2,Almería,249,8775,Andalucía,Almería,Almería,770554,2.257500e+08,87.81,8774.0,4,AL,Diputación de Almería,Edificio de la Diputación Provincial de Almerí...,168886,17.3,14185706,15159388,15282092,16003582,14764342,14861150,16494994,18250223,28.65,1.57,1.730
3,Araba/Álava,NaN,3037,País Vasco,Vitoria,Vitoria,341961,4.346069e+08,112.60,NaN,1,VI,Diputación Foral de Álava,Palacio de la Diputación Foral de Álava (Vitoria),36831,9.1,11960737,12336129,12721159,12797528,11764228,11958641,12838267,14090604,17.81,0.70,0.600
4,Asturias,401,10 604,Principado de Asturias,Oviedo,Gijón,1015128,2.919000e+08,95.73,10603.0,33,O,No tiene[1]​,No tiene[1]​,58387,12.5,21675441,22538267,23185680,23666618,21376392,23617641,26690102,28350371,30.79,2.07,2.100
5,Badajoz,NaN,21 766,Extremadura,Badajoz,Badajoz,665155,1.225658e+08,30.56,NaN,6,BA,Diputación de Badajoz,NaN,25469,6.6,11776505,12419839,12609609,12838980,11979224,12729424,13644123,14927003,26.75,1.35,4.300
6,Barcelona,161,7728,Cataluña,Barcelona,Barcelona,5959941,3.595900e+09,771.21,7733.0,8,B,Diputación de Barcelona,Casa Serra (Barcelona),1015499,22.3,158516150,166618005,173436258,180560179,161885097,173917445,191251096,209563285,32.20,12.13,1.530
7,Bizkaia,154,2217,País Vasco,Bilbao,Bilbao,1167233,6.635000e+08,526.49,2217.0,48,BI,Diputación Foral de Vizcaya,Palacio de la Diputación Foral de Vizcaya (Bil...,105893,14.6,33686330,34314272,35599877,36737656,33016596,35910195,41009633,44229902,31.30,2.38,0.440
8,Burgos,NaN,14 292,Castilla y León,Burgos,Burgos,362663,2.200000e+08,25.38,NaN,9,BU,Diputación de Burgos,NaN,35158,8.2,9572090,10054308,10494196,10628487,9620320,10224118,10709841,11303996,18.09,0.74,2.820
9,Cantabria,284,5321,Cantabria,Santander,Santander,593623,2.236000e+08,111.56,5325.0,39,S,No tiene[4]​,No tiene[4]​,44242,9.8,12833899,13318023,13843935,14274351,12991241,14206538,15557155,16740663,29.93,1.21,1.050
